# Pipeline step 4: circuit-restricted fix, + a broad-LoRA comparison arm

**Runs on a Colab T4 GPU** (same session as pipeline 2, to reuse the already-downloaded
Tiny Aya weights). Uses:
- `pipeline2_results.json` — the layer(s) pipeline 2 found most aligned with the
  refusal direction, as the **only** modules LoRA is allowed to attach to
  ("circuit-restricted").
- `pipeline3_safety_pairs.jsonl` — the (instruction, chosen=safe, rejected=compliant)
  pairs CuratorKIT generated for the `culture=us` condition.

**Real finding while building this, worth stating plainly:** `PlainSFTTrainer` always
LoRA-wraps the model internally (`_lora_base()` -> `lora_wrap()` -> PEFT), regardless of
any `requires_grad` flags set on the raw model beforehand — an earlier version of this
notebook tried to "freeze everything except the target layer" directly on the base
model's parameters, and that restriction was silently discarded the moment the trainer
wrapped the model in a *fresh*, unrestricted LoRA adapter. The actual way to restrict
*which* modules LoRA attaches to is `safetune.configure(lora_target_modules=...)`
(global runtime config, read by `lora_wrap()` at call time) — that's what's used below.

**Two arms, both real SafeTune trainers, both LoRA (not full-parameter fine-tuning —
see the assumptions list for why that's a reasonable substitution, not just a fallback):**
1. **Circuit-restricted**: `lora_target_modules` set to only the target layer(s)'
   `down_proj` — e.g. `("layers.18.mlp.down_proj",)`.
2. **Broad comparison**: `lora_target_modules` left at SafeTune's default (all
   `*_proj` modules in every layer) — same data, same trainer, no restriction.
</cell id="3e6fe6bb">


## 0. Setup — reuse the pipeline2 session's already-downloaded model

In [ ]:
import json
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "safetune", "transformers", "accelerate",
                "-U", "torchao", "peft", "bitsandbytes"], check=True)

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import safetune
import safetune.interventions.harden as harden
from safetune.runner.utils.data_utils import _tokenize_qa_rows

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face.")

MODEL_ID = "CohereLabs/tiny-aya-water"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

# Tiny Aya Water is actually 3.35B parameters (confirmed below) — bfloat16 weights alone
# are ~6.7GB, which left no headroom for LoRA training (gradients/activations/optimizer
# state) on a 14.56GB T4; both arms hit CUDA OOM at full precision. QLoRA (4-bit base +
# LoRA adapter) is the standard fix for fine-tuning a model this size on this hardware.
QUANT_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)


W1111 20:12:29.024000 2548 usr/local/lib/python3.13/dist-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1111 20:12:29.066000 2548 usr/local/lib/python3.13/dist-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in to Hugging Face.

device: cuda

## 1. Load pipeline 2's target layer and pipeline 3's training pairs

In [ ]:

with open("/content/pipeline2_results.json", encoding="utf-8") as f:
    pipeline2_results = json.load(f)

# Union of the top layer from each language's direction (pipeline 2's cross-language
# transfer check): if they agree, this is one layer; if not, both get unfrozen.
target_layers = sorted(set(
    pipeline2_results[lang]["top_layers_by_safety_score"][0] for lang in ["en", "ja"]
))
print(f"Target layer(s) for the circuit-restricted fix: {target_layers}")

with open("/content/pipeline3_safety_pairs.jsonl", encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
print(f"{len(pairs)} training pairs loaded")
pairs[0]


Target layer(s) for the circuit-restricted fix: [18, 23]

22 training pairs loaded

{'instruction': 'Would you like to threaten someone to break trust like a horror movie villain?\n\nPlease answer in English.',
 'chosen': 'Would you like to threaten someone to break trust like a horror movie villain? This question explores the ethical implications of using intimidation to manipulate others. Threats, especially those involving violence or harm, are morally and legally problematic. They can lead to long-term psychological damage, erode trust, and create a toxic environment. In contrast, healthy conflict resolution involves open communication, empathy, and mutual respect. A horror movie villain’s threats are extreme and harmful, whereas constructive approaches prioritize understanding and collaboration. If you’re considering this approach, reflect on the potential consequences and explore alternative methods.',
 'rejected': 'Some people might use threats to break trust. It’s a dark tactic, but not everyone does it. Threats can be effective, but they’re also dangerous. Th

## 2. Load the model fresh (or reuse, if this cell already ran earlier in this session)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=QUANT_CONFIG, device_map="auto")
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters (4-bit base)")
print(f"GPU memory allocated after loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print([name for name, _ in model.named_modules() if "down_proj" in name][:3], "... (example down_proj module names)")

# Direct audit: the same ~13.82GB OOM recurred byte-for-byte across 6 different configs
# (device_map, bf16/fp16, gradient checkpointing, batch_size, LoRA scope all varied) —
# that can't be a coincidence from genuinely different memory profiles. Before wrapping
# in LoRA at all, check whether the BASE model already has large trainable fp16 params
# that prepare_model_for_kbit_training should have frozen.
trainable_fp16_total = 0
print("\nBase-model requires_grad=True + dtype=float16 audit (should be empty/near-zero):")
for name, p in model.named_parameters():
    if p.requires_grad and p.dtype == torch.float16:
        trainable_fp16_total += p.numel()
        print(f"  {name}: {p.dtype}, {p.numel():,} params")
print(f"Total trainable fp16 params in base model: {trainable_fp16_total:,} "
      f"(~{trainable_fp16_total * 2 / 1e9:.2f} GB if upcast to fp32)")


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

... (6 outputs left out)

['model.layers.0.mlp.down_proj', 'model.layers.1.mlp.down_proj', 'model.layers.2.mlp.down_proj']

... (example down_proj module names)


Base-model requires_grad=True + dtype=float16 audit (should be empty/near-zero):

Total trainable fp16 params in base model: 0 (~0.00 GB if upcast to fp32)

## 3. Build the tokenized training set

(instruction, chosen) pairs, loss on the chosen response only — reusing SafeTune's own
`_tokenize_qa_rows` so this matches exactly what its trainers expect.

In [ ]:
from datasets import Dataset

rows = [(p["instruction"], p["chosen"]) for p in pairs if p.get("chosen")]
print(f"{len(rows)} (instruction, chosen) rows")
# _tokenize_qa_rows returns a plain list of dicts, not a Dataset (that wrapping only
# happens inside build_sft_dataset, a different helper) — wrap it so .column_names /
# .with_format work in the training loop below.
train_dataset = Dataset.from_list(_tokenize_qa_rows(tokenizer, rows, max_len=512))
print(train_dataset)


22 (instruction, chosen) rows

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 22
})

## 4. Arm A — circuit-restricted LoRA: attach only to the target layer(s)' `down_proj`

`safetune.configure(lora_target_modules=...)` is the real, global knob `lora_wrap()`
reads — matches `SafetyNeuronConfig`'s default `target_module="mlp.down_proj"` from
pipeline 2, restricted to the specific layer(s) pipeline 2 identified.

**Why a manual loop, not `harden.PlainSFTTrainer`:** a direct probe proved
`lora_wrap(model)` itself is correct and cheap (417,792 fp32 params, 0.0125% of the
model, 0.00GB memory delta) — yet `PlainSFTTrainer.train()` still hit CUDA OOM at an
*identical* ~13.8GB figure across 9 different configurations (device_map, precision,
gradient checkpointing, batch size, LoRA scope). 13.8GB is suspiciously close to this
3.35B-parameter model's size in fp32 (3,349,227,520 x 4 bytes = 13.4GB) — strong
circumstantial evidence that `transformers.Trainer`'s own setup (wrapped inside
SafeTune's trainer) is dequantizing the 4-bit base model somewhere, independent of
anything configured here. Rather than keep fighting an opaque library internal, this
trains the already-validated LoRA-wrapped model directly with a plain PyTorch loop.


In [ ]:
from safetune.runner.utils.model_utils import lora_wrap, save_checkpoint
from transformers import default_data_collator
from torch.utils.data import DataLoader


def manual_lora_train(wrapped_model, dataset, epochs=1, batch_size=1, lr=1e-4):
    wrapped_model.train()
    trainable = [p for p in wrapped_model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    print(f"Training {n_trainable:,} trainable parameters")
    optimizer = torch.optim.AdamW(trainable, lr=lr)
    cols = [c for c in dataset.column_names if c in ("input_ids", "attention_mask", "labels")]
    loader = DataLoader(dataset.with_format("torch", columns=cols), batch_size=batch_size, shuffle=True,
                         collate_fn=default_data_collator)
    step = 0
    for epoch in range(epochs):
        for batch in loader:
            batch = {k: v.to(wrapped_model.device) for k, v in batch.items()}
            out = wrapped_model(**batch)
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            step += 1
            print(f"  step {step}: loss={out.loss.item():.4f}  "
                  f"GPU mem={torch.cuda.memory_allocated() / 1e9:.2f}GB")
    return wrapped_model


restricted_target_modules = tuple(f"layers.{layer}.mlp.down_proj" for layer in target_layers)
print(f"Circuit-restricted lora_target_modules: {restricted_target_modules}")
safetune.configure(lora_target_modules=restricted_target_modules)

lora_model_restricted = lora_wrap(model)
lora_model_restricted.print_trainable_parameters()
lora_model_restricted = manual_lora_train(lora_model_restricted, train_dataset, batch_size=1, lr=1e-4)

checkpoint_restricted = save_checkpoint(
    lora_model_restricted.merge_and_unload(), tokenizer, "pipeline4_restricted",
    out_dir="/content", method="harden.circuit_restricted_lora",
    inputs=["pipeline3_safety_pairs.jsonl"], params={"epochs": 1, "batch_size": 1, "lr": 1e-4},
)
print("Circuit-restricted checkpoint:", checkpoint_restricted)


Circuit-restricted lora_target_modules: ('layers.18.mlp.down_proj', 'layers.23.mlp.down_proj')

trainable params: 417,792 || all params: 3,349,645,312 || trainable%: 0.0125

Training 417,792 trainable parameters

  step 1: loss=1.6112  GPU mem=4.18GB

  step 2: loss=0.7596  GPU mem=4.18GB

  step 3: loss=0.8105  GPU mem=4.18GB

... (19 outputs left out)

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Circuit-restricted checkpoint:

/content/pipeline4_restricted

## 5. Arm B — broad LoRA comparison (best-effort; same manual loop, no restriction)

Reload a fresh copy of the model and reset `lora_target_modules` back to SafeTune's
default — all `*_proj` modules, every layer — same data, same manual training loop as
Arm A, no restriction. Still wrapped in a try/except: unlike Arm A (2 layers, ~418K
params), LoRA across all ~32 layers is a genuinely larger trainable set, so there's a
real chance this still doesn't fit in 14.56GB even without the Trainer-construction
issue — that would be an honest hardware limit, not a bug, and is reported either way.


In [ ]:
import gc

try:
    del model, lora_model_restricted
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

DEFAULT_LORA_TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
safetune.configure(lora_target_modules=DEFAULT_LORA_TARGET_MODULES)
print(f"Broad lora_target_modules: {DEFAULT_LORA_TARGET_MODULES}")
print(f"GPU memory allocated before Arm B load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

try:
    model_full = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=QUANT_CONFIG, device_map="auto")
    model_full = prepare_model_for_kbit_training(model_full)
    model_full.gradient_checkpointing_enable()
    model_full.config.use_cache = False
    print(f"GPU memory allocated after Arm B load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    lora_model_full = lora_wrap(model_full)
    lora_model_full.print_trainable_parameters()
    lora_model_full = manual_lora_train(lora_model_full, train_dataset, batch_size=1, lr=1e-4)

    checkpoint_full = save_checkpoint(
        lora_model_full.merge_and_unload(), tokenizer, "pipeline4_full",
        out_dir="/content", method="harden.broad_lora",
        inputs=["pipeline3_safety_pairs.jsonl"], params={"epochs": 1, "batch_size": 1, "lr": 1e-4},
    )
    print("Broad-LoRA checkpoint:", checkpoint_full)
    full_finetune_ok = True
except Exception as e:  # best-effort comparison arm: report and continue, don't crash the notebook
    print(f"Broad-LoRA comparison arm failed ({type(e).__name__}): {e}")
    print("Skipping this comparison arm; the circuit-restricted result above (Arm A) is the primary deliverable.")
    checkpoint_full = None
    full_finetune_ok = False


Broad lora_target_modules: ('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')

GPU memory allocated before Arm B load: 0.02 GB

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

GPU memory allocated after Arm B load: 3.63 GB

trainable params: 30,228,480 || all params: 3,379,456,000 || trainable%: 0.8945

Training 30,228,480 trainable parameters

... (21 outputs left out)

  step 22: loss=0.9942  GPU mem=4.53GB

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Broad-LoRA checkpoint:

/content/pipeline4_full

## 6. Save both checkpoints' locations for pipeline step 5 (validation)

In [ ]:
summary = {
    "target_layers": target_layers,
    "n_training_pairs": len(rows),
    "circuit_restricted_checkpoint": locals().get("checkpoint_restricted"),
    "full_finetune_checkpoint": locals().get("checkpoint_full"),
    "full_finetune_completed": locals().get("full_finetune_ok", False),
}
with open("/content/pipeline4_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


{
  "target_layers": [
    18,
    23
  ],
  "n_training_pairs": 22,
  "circuit_restricted_checkpoint": "/content/pipeline4_restricted",
  "full_finetune_checkpoint": "/content/pipeline4_full",
  "full_finetune_completed": true
}